In [0]:
df = display(spark.sql("""
SELECT *
FROM mvn_db_catalog.bronze.health_claims
"""))

In [0]:
df = spark.table("mvn_db_catalog.bronze.health_claims")

In [0]:
Silver_df = df.dropDuplicates(["ClaimID"])
display(Silver_df)

In [0]:
from pyspark.sql.functions import col
Silver_df = df.filter(col("ClaimDate").isNotNull())

In [0]:
df = df.filter(
    col("Hospital").rlike("Hospital$")
)

In [0]:
from pyspark.sql.functions import to_date
df = (
    df.withColumn("ClaimID", col("ClaimID").cast("int"))
      .withColumn("ClaimAmount", col("ClaimAmount").cast("double"))
      .withColumn("ClaimDate", to_date("ClaimDate"))
)

In [0]:
valid_status = ["Approved", "Rejected", "Pending"]

df = df.filter(col("Status").isin(valid_status))

In [0]:
%sql
USE CATALOG mvn_db_catalog;

CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
(
    df.write
      .format("delta")
      .mode("overwrite")
      .saveAsTable("mvn_db_catalog.silver.health_claims")
)

In [0]:
%sql
SELECT *
FROM mvn_db_catalog.silver.health_claims;

In [0]:
silver_path = "abfss://silver@mvnazureproject.dfs.core.windows.net/"

(
    Silver_df.write
        .format("delta")
         .option("inferSchema", True)
        .mode("overwrite")\
            .option("overwriteSchema", "true") 
        .save(silver_path)
)

In [0]:
display(
    Silver_df,
    checkpointLocation="abfss://bronze@mvnazureproject.dfs.core.windows.net/delta"
)

In [0]:
%sql
SELECT *
FROM mvn_db_catalog.bronze.health_claims
WHERE ClaimID >= 1008
ORDER BY ClaimID;